In [1]:
import cv2
import numpy as np
import mediapipe as mp
import pickle
from pathlib import Path
from tqdm import tqdm

# ================= CONFIG =================
DATASET_DIR = r"D:\Users\Anoshia\BattingEdge_FYP\v9\dataset"
OUTPUT_DIR  = r"D:\Users\Anoshia\BattingEdge_FYP\v9_5\features"
SEQUENCE_LENGTH = 50
# ==========================================

mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

def calculate_angle(a, b, c):
    a = np.array([a.x, a.y])
    b = np.array([b.x, b.y])
    c = np.array([c.x, c.y])

    ba = a - b
    bc = c - b

    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))

def process_video(video_path):
    cap = cv2.VideoCapture(str(video_path))
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(rgb)

        if not results.pose_landmarks:
            continue

        lm = results.pose_landmarks.landmark

        # --- 99 RAW POSE FEATURES ---
        pose_feats = []
        for p in lm:
            pose_feats.extend([p.x, p.y, p.z])

        # --- 8 BIOMECHANICAL ANGLES ---
        try:
            left_elbow  = calculate_angle(lm[11], lm[13], lm[15])
            right_elbow = calculate_angle(lm[12], lm[14], lm[16])

            left_knee  = calculate_angle(lm[23], lm[25], lm[27])
            right_knee = calculate_angle(lm[24], lm[26], lm[28])

            shoulder_angle = np.degrees(np.arctan2(
                lm[12].y - lm[11].y,
                lm[12].x - lm[11].x
            ))

            hip_angle = np.degrees(np.arctan2(
                lm[24].y - lm[23].y,
                lm[24].x - lm[23].x
            ))

            bat_angle = np.degrees(np.arctan2(
                lm[15].y - lm[11].y,
                lm[15].x - lm[11].x
            ))

            stance_width = abs(lm[27].x - lm[28].x)

            bio = [
                left_elbow, right_elbow,
                left_knee, right_knee,
                shoulder_angle, hip_angle,
                bat_angle, stance_width
            ]

        except:
            bio = [0.0] * 8

        frames.append(pose_feats + bio)

    cap.release()

    if len(frames) < 10:
        return None

    frames = np.array(frames)

    # --- TEMPORAL RESAMPLING ---
    resampled = np.zeros((SEQUENCE_LENGTH, frames.shape[1]))
    x_old = np.linspace(0, len(frames) - 1, len(frames))
    x_new = np.linspace(0, len(frames) - 1, SEQUENCE_LENGTH)

    for i in range(frames.shape[1]):
        resampled[:, i] = np.interp(x_new, x_old, frames[:, i])

    return resampled

# ================= RUN =================
out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)

for split in ["train", "val", "test"]:
    X, y = [], []
    split_dir = Path(DATASET_DIR) / split

    classes = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])

    if split == "train":
        with open(out / "classes.pkl", "wb") as f:
            pickle.dump(classes, f)

    for idx, cls in enumerate(classes):
        for vid in tqdm((split_dir / cls).glob("*.mp4"), desc=f"{split}/{cls}"):
            feat = process_video(vid)
            if feat is not None:
                X.append(feat)
                y.append(idx)

    np.save(out / f"X_{split}.npy", np.array(X))
    np.save(out / f"y_{split}.npy", np.array(y))
    print(f"✅ Saved {split}: {len(X)} samples")

print("✅ V9.5 FEATURE EXTRACTION COMPLETE")


train/Cover Drive: 0it [00:00, ?it/s]d:\Users\Anoshia\BattingEdge_FYP\venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
train/Cover Drive: 754it [52:40,  4.19s/it]
train/Cut Shot: 879it [42:28,  2.90s/it]
train/Defense: 761it [1:08:18,  5.39s/it]
train/Pull Shot: 742it [48:06,  3.89s/it]
train/Sweep Shot: 754it [51:05,  4.07s/it]


✅ Saved train: 3007 samples


val/Cover Drive: 102it [06:44,  3.97s/it]
val/Cut Shot: 112it [06:02,  3.24s/it]
val/Defense: 97it [09:40,  5.98s/it]
val/Pull Shot: 94it [06:56,  4.44s/it]
val/Sweep Shot: 88it [06:20,  4.33s/it]


✅ Saved val: 388 samples


test/Cover Drive: 99it [07:14,  4.39s/it]
test/Cut Shot: 115it [07:11,  3.75s/it]
test/Defense: 99it [08:02,  4.88s/it]
test/Pull Shot: 100it [05:54,  3.54s/it]
test/Sweep Shot: 89it [06:41,  4.51s/it]

✅ Saved test: 378 samples
✅ V9.5 FEATURE EXTRACTION COMPLETE
